<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----
Уведомления

### Вариант задания  № 24


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Notification в C#, который будет представлять уведомления
пользователям. На основе этого класса разработать 2-3 производных класса,
демонстрирующих принципы наследования и полиморфизма. В каждом из классов
должны быть реализованы новые атрибуты и методы, а также переопределены
некоторые методы базового класса для демонстрации полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
// Интерфейс для уведомлений
public interface INotification
{
    void SendNotification();
    string GetNotificationDetails();
    DateTime GetCreationDate();
    bool Validate();
}

public abstract class Notification : INotification
{
    public int NotificationID { get; set; }
    public string MessageText { get; set; }
    public string Type { get; set; }
    public DateTime CreatedAt { get; set; }
    public int Priority { get; set; }
    public bool IsSent { get; private set; }
    public string Language { get; set; }
    public int RetryCount { get; private set; }

    protected Notification(int id, string message, string type)
    {
        NotificationID = id;
        MessageText = message;
        Type = type;
        CreatedAt = DateTime.Now;
        Priority = 1;
        Language = "RU";
        RetryCount = 0;
    }

    // Явная реализация интерфейса
    DateTime INotification.GetCreationDate() => CreatedAt;
    
    bool INotification.Validate() => !string.IsNullOrEmpty(MessageText);

    public virtual void SendNotification()
    {
        Console.WriteLine($"Отправка {Type} уведомления...");
        IsSent = true;
    }

    public virtual void DisplayNotification()
    {
        Console.WriteLine($"Уведомление [{NotificationID}]: {MessageText}");
    }

    public virtual string GetNotificationDetails()
    {
        return $"ID: {NotificationID}, Тип: {Type}, Сообщение: {MessageText}, Приоритет: {Priority}, Язык: {Language}";
    }

    // Новые методы
    public void MarkAsSent() => IsSent = true;
    
    public void SetPriority(int priority) => Priority = Math.Clamp(priority, 1, 5);
    
    public void Retry()
    {
        RetryCount++;
        Console.WriteLine($"Повторная отправка #{RetryCount}");
    }
    
    public TimeSpan GetAge() => DateTime.Now - CreatedAt;
}

public class EmailNotification : Notification
{
    public string EmailAddress { get; set; }
    public string Subject { get; set; }
    public string Format { get; set; }
    public List<string> Attachments { get; set; }

    public EmailNotification(int id, string message, string email) : base(id, message, "Email")
    {
        EmailAddress = email;
        Subject = "Уведомление";
        Format = "HTML";
        Attachments = new List<string>();
    }

    public override void SendNotification()
    {
        base.SendNotification();
        Console.WriteLine($"Отправлено сообщение на email: {EmailAddress}");
        Console.WriteLine($"Тема: {Subject}, Формат: {Format}");
        if (Attachments.Any())
            Console.WriteLine($"Вложения: {string.Join(", ", Attachments)}");
    }

    public override string GetNotificationDetails()
    {
        return base.GetNotificationDetails() + $", Email: {EmailAddress}, Тема: {Subject}";
    }

    // Новые методы
    public void AddAttachment(string file) => Attachments.Add(file);
    
    public void ValidateEmail() => Console.WriteLine(EmailAddress.Contains("@") ? 
        "Email валиден" : "Неверный формат email");
}

public class SMSNotification : Notification
{
    public string PhoneNumber { get; set; }
    public string Provider { get; set; }
    public bool IsUnicode { get; set; }
    public int MaxLength { get; set; }

    public SMSNotification(int id, string message, string number) : base(id, message, "SMS")
    {
        PhoneNumber = number;
        Provider = "Default";
        IsUnicode = false;
        MaxLength = 160;
    }

    public override void SendNotification()
    {
        base.SendNotification();
        Console.WriteLine($"Отправлено SMS на номер: {PhoneNumber}");
        Console.WriteLine($"Провайдер: {Provider}, Юникод: {IsUnicode}");
    }

    public override string GetNotificationDetails()
    {
        return base.GetNotificationDetails() + $", Номер телефона: {PhoneNumber}";
    }

    // Новые методы
    public void SplitMessage()
    {
        var parts = (int)Math.Ceiling(MessageText.Length / (double)MaxLength);
        Console.WriteLine($"Сообщение разделено на {parts} частей");
    }
    
    public void SetProvider(string provider) => Provider = provider;
}

public class PushNotification : Notification
{
    public string Platform { get; set; }
    public string DeviceToken { get; set; }
    public int ExpiryHours { get; set; }
    public string Sound { get; set; }

    public PushNotification(int id, string message, string platform) : base(id, message, "Push")
    {
        Platform = platform;
        DeviceToken = Guid.NewGuid().ToString();
        ExpiryHours = 24;
        Sound = "default";
    }

    public override void DisplayNotification()
    {
        Console.WriteLine($"[{Platform}] Push-уведомление: {MessageText}");
    }

    public override void SendNotification()
    {
        base.SendNotification();
        Console.WriteLine($"Отправка push-уведомления на платформу {Platform}");
        Console.WriteLine($"Токен устройства: {DeviceToken}, Срок действия: {ExpiryHours}ч");
    }

    public override string GetNotificationDetails()
    {
        return base.GetNotificationDetails() + $", Платформа: {Platform}";
    }

    // Новые методы
    public void SetExpiry(int hours) => ExpiryHours = hours;
    
    public bool IsExpired() => GetAge().TotalHours > ExpiryHours;
}

// Сервис для управления зависимостями
public class NotificationService
{
    private readonly List<Notification> _notifications;

    public NotificationService()
    {
        _notifications = new List<Notification>();
    }

    public void AddNotification(Notification notification)
    {
        _notifications.Add(notification);
    }

    public void ProcessAllNotifications()
    {
        foreach (var notification in _notifications)
        {
            Console.WriteLine("\n--- Обработка уведомления ---");
            notification.DisplayNotification();
            
            // Использование явной реализации интерфейса
            var iNotification = (INotification)notification;
            if (iNotification.Validate())
            {
                notification.SendNotification();
                Console.WriteLine($"Создано: {iNotification.GetCreationDate()}");
            }
            else
            {
                Console.WriteLine("Ошибка валидации уведомления!");
            }
            
            Console.WriteLine("Детали: " + notification.GetNotificationDetails());
            
            // Демонстрация новых методов
            if (notification is EmailNotification email)
                email.ValidateEmail();
            else if (notification is SMSNotification sms)
                sms.SplitMessage();
            else if (notification is PushNotification push)
                Console.WriteLine($"Просрочено: {push.IsExpired()}");
        }
    }
}

        Console.WriteLine("Демонстрация системы уведомлений:");
        Console.WriteLine("==================================");

        var service = new NotificationService();
        
        // Создание уведомлений
        var email = new EmailNotification(1, "Добро пожаловать в нашу систему!", "user@example.com");
        email.SetPriority(3);
        email.AddAttachment("manual.pdf");
        service.AddNotification(email);

        var sms = new SMSNotification(2, "Ваш код подтверждения: 1234", "+79991234567");
        sms.SetProvider("Megafon");
        sms.SetPriority(5);
        service.AddNotification(sms);

        var push = new PushNotification(3, "У вас новое сообщение", "Android");
        push.SetExpiry(1);
        service.AddNotification(push);

        // Обработка через сервис
        service.ProcessAllNotifications();

        Console.WriteLine("\n==================================");
        Console.WriteLine("Обработка всех уведомлений завершена!");


Демонстрация системы уведомлений:

--- Обработка уведомления ---
Уведомление [1]: Добро пожаловать в нашу систему!
Отправка Email уведомления...
Отправлено сообщение на email: user@example.com
Тема: Уведомление, Формат: HTML
Вложения: manual.pdf
Создано: 11/3/2025 12:26:16 AM
Детали: ID: 1, Тип: Email, Сообщение: Добро пожаловать в нашу систему!, Приоритет: 3, Язык: RU, Email: user@example.com, Тема: Уведомление
Email валиден

--- Обработка уведомления ---
Уведомление [2]: Ваш код подтверждения: 1234
Отправка SMS уведомления...
Отправлено SMS на номер: +79991234567
Провайдер: Megafon, Юникод: False
Создано: 11/3/2025 12:26:16 AM
Детали: ID: 2, Тип: SMS, Сообщение: Ваш код подтверждения: 1234, Приоритет: 5, Язык: RU, Номер телефона: +79991234567
Сообщение разделено на 1 частей

--- Обработка уведомления ---
[Android] Push-уведомление: У вас новое сообщение
Отправка Push уведомления...
Отправка push-уведомления на платформу Android
Токен устройства: 4bece57c-e9c6-416d-b9bc-eecced0c3558, 